# SO / Planck / LiteBIRD comparison + unified constraints

This notebook can either **load** precomputed Fisher pickles or **recompute** them.

Pickles:
- `../fisher_data/so_fisher_all.pkl` (SO + PlanckLite fishers)
- `../fisher_data/lb_fisher_10.pkl` (LiteBIRD fishers)

By default it will **not recompute** if the files already exist.


In [1]:
# --- environment helpers (HPC-friendly) ---
import os
import sys
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/mplcache')
# If `classy`/`cobaya`/`getdist` live in a user site-packages, add it here.
extra_site = Path('/home/sa5705/.local/soconda_v0.6.8/lib/python3.12/site-packages')
if extra_site.exists() and str(extra_site) not in sys.path:
    sys.path.insert(0, str(extra_site))


In [2]:
import os
from pathlib import Path

# Keep matplotlib caches out of $HOME
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

import utils


In [3]:
import sys
sys.path.append("../")
sys.path.append("../cosmocast_makelik/")
sys.path.append("../cosmocast_makelik/multi_freq_liq")

In [4]:
# --- paths + recompute switches ---
# Set these True only when you explicitly want to regenerate the pickles.
RECOMPUTE_SO_FISHER_ALL = False
RECOMPUTE_LB_FISHERS    = False

SO_FISHER_PATH = Path('..') / 'fisher_data' / 'so_fisher_all.pkl'
LB_FISHER_PATH = Path('..') / 'fisher_data' / 'lb_fisher_10.pkl'


In [5]:
# --- (optional) recompute SO + PlanckLite fisher pickle ---
# Requires `classy` (CLASS) and `cobaya`.

if SO_FISHER_PATH.exists() and not RECOMPUTE_SO_FISHER_ALL:
    print('Found', SO_FISHER_PATH, '- not recomputing.')
else:
    out = utils.compute_so_fisher_all(output_path=SO_FISHER_PATH)
    print('Wrote', out)


Wrote ../fisher_data/so_fisher_all.pkl


In [6]:
# --- (optional) recompute LiteBIRD fisher pickle ---
# Requires `classy` (CLASS). If CLASS isn't available in your kernel,
# leave RECOMPUTE_LB_FISHERS=False and use the precomputed pickle.

if LB_FISHER_PATH.exists() and not RECOMPUTE_LB_FISHERS:
    print('Found', LB_FISHER_PATH, '- not recomputing.')
else:
    utils.compute_litebird_fisher_grids(output_path=LB_FISHER_PATH)


In [7]:
# --- load pickles (after optional recompute) ---

theta_full = utils.THETA_FULL
so_fisher_all = utils.load_so_fisher_all(SO_FISHER_PATH)
muci = utils.load_litebird_grids(LB_FISHER_PATH)

corr_plot = ['pcor', 'ucor', 'acor']
iso_plot = ['cdi', 'nid', 'niv']
year_tag = '10yr'


In [8]:
# --- compute + save Planck-only / SO-only / LiteBIRD-only / unified constraints ---

utils.compare_and_save_constraints(
    so_fisher_all=so_fisher_all,
    muci=muci,
    theta_full=theta_full,
    corr_plot=corr_plot,
    iso_plot=iso_plot,
    year_tag=year_tag,
    out_base='images/unified_constraints',
    skip_existing=True,
    write_tables=True,
    write_corr=True,
    write_triangles=True,
    write_overlays=True,
)

print('Wrote outputs under:', Path('images/unified_constraints') / year_tag)


Wrote outputs under: images/unified_constraints/10yr


## Additional (non-Fisher) plots

These reproduce the other plot types from the original script:
- unified noise curves (TT + EE)
- covariance-diagonal diagnostic plots (TT, EE, TE)

They **require** `classy` (CLASS) and `cobaya` (PlikLite loader).


In [9]:
import nonfisher_plots

RECOMPUTE_NONFISHER_PLOTS = False
NONFISHER_OUTDIR = Path('images/non_fisher')

# If outputs already exist and recompute is False, do nothing.
# (Files are written under images/non_fisher/y<year>/)
if RECOMPUTE_NONFISHER_PLOTS:
    nonfisher_plots.compute_and_save_noise_and_cov_plots(
        out_dir=NONFISHER_OUTDIR,
        year=5,
        fsky_lat=0.4,
        skip_existing=True,
    )
else:
    print('Skipping non-Fisher plot generation (set RECOMPUTE_NONFISHER_PLOTS=True to run).')


Skipping non-Fisher plot generation (set RECOMPUTE_NONFISHER_PLOTS=True to run).


In [10]:
# Quick sanity check: one case's unified sigma table
corr='pcor'
iso='cdi'

pk = so_fisher_all[corr]['PK_Lite'][iso][year_tag]
so = so_fisher_all[corr]['SO'][iso][year_tag]
lb = muci[corr]['LB'][iso][year_tag]

unified = pk.combine(so).combine(lb)
unified.summary_table(theta_full[corr]['theta0'], scaled_params={'P_RR_1','P_RR_2','P_II_1','P_II_2'})


,Fiducial,sigma,S/N
omega_b,0.02237,0.000025,891.256879
omega_cdm,0.11933,0.000098,1220.732952
h,0.67660,0.000507,1335.710407
tau_reio,0.05610,0.001549,36.213490
P_RR_1,23.00000,0.130485,176.266006
P_RR_2,23.00000,0.069501,330.929858
P_II_1,15.00000,0.445069,33.702622
P_II_2,15.00000,0.765254,19.601335
